# Unit 12 solutions: tuning against race time

These reference solutions measure the objective's noise, map three knobs, run a small evolution strategy and a random search, check the lesson's training win on the validation track, apply the crash-aware check and Unit 11's margin rule, profile a race, and walk into the rounding traps on purpose; the Challenges search the speed loop and race a policy gradient against coordinate search.
Each exercise's statement is repeated, followed by a prediction made before running anything, then the solution.
Every race is seeded and uses `MPC_BUDGET_MS = None`, so every race number below is reproducible, and executable asserts check each answer; the profiler's timings depend on the machine and are printed, never asserted.
Every number is a number about racekit's **carla-like car**: a model, not the truth.
**No race here touches `test-circuit-seed2028`**, the unit's test track.

In [ ]:
import cProfile
import math
import pstats
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_11, UNIT_12 = AGENTS / "unit-11.py", AGENTS / "unit-12.py"
workdir = Path(mkdtemp(prefix="roar-unit-12-sol-"))
TICK = 0.05 + 1e-9  # s: one control tick (race times are multiples of it)
DT = 0.05  # s: one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
NO_TIMING = {"MPC_BUDGET_MS": None}  # every race here: no wall clock inside the race
VALIDATION = "test-circuit-seed2026"  # the unit's validation track
# Unit 8's grip probes (m/s^2 per grip section, rounded to 0.01); test-circuit's ship in the agents.
MEASURED_MU_G = {
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "test-circuit-seed2026": (7.70, 7.69, 7.70, 9.81, 8.16, 8.04),
}


# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


# The lines: test-circuit's as the agents ship it, the others solved here.
with shim_installed():
    Unit11, _ = load_submission(UNIT_11)
control, line_offsets = {}, {}
for name in ("test-circuit", "test-circuit-seed1", VALIDATION):
    track = load_track(name)
    if name == "test-circuit":
        line_offsets[name] = np.array(Unit11(track.waypoints, None).offsets)
    else:
        control[name] = solve_line(track)
        line_offsets[name] = spline_basis(len(track)) @ control[name]


written = []  # every variant file written so far


def variant(source, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, each written as its repr.
    path = workdir / f"v{len(written):03d}.py"
    written.append(path)
    return write_variant(source, path, {key: repr(value) for key, value in constants.items()})


def on_track(track_name, **constants):
    # The constants a race on this track needs: its own line and measured grip (test-circuit: as shipped).
    if track_name != "test-circuit":
        constants = {"LINE_CONTROL": tuple(float(value) for value in control[track_name]),
                     "MU_G": MEASURED_MU_G[track_name], **constants}
    return constants


def race(setting, track="test-circuit", seed=0, laps=1, source=UNIT_11, k=None, max_seconds=450.0):
    # One race of `source` with the constants in `setting` changed and the timing off, on `track` with its line
    # and grip, on the carla-like car with its noise (car k of Unit 11's perturbed family if k is given).
    path = variant(source, **on_track(track, **setting), **NO_TIMING)
    return run_race(path, track, car=carla_like(noise=True, perturb=k), seed=seed, laps=laps, max_seconds=max_seconds)


def line_error(result):
    # The largest |cross-track error from the race's own line| over the driving rows (held == 0).
    offsets = line_offsets[result.track]
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result):
    return result.finished and result.collisions == 0 and line_error(result) <= CLEAN_ERROR


def same_race(a, b):
    # The same race: every telemetry row equal, not only the time.
    return len(a.telemetry) == len(b.telemetry) and all(
        np.sum(a.telemetry[name] != b.telemetry[name]) == 0 for name in a.telemetry.dtype.names)

## Exercise 1 - The objective's noise

How much does a race time move when nothing changes but the race seed?
Race `agents/unit-11.py` (timing off: `variant(UNIT_11, **NO_TIMING)`) five times on `test-circuit`, 1 lap, race seeds 0 to 4, with `repeat_races(source, "test-circuit", n=5, seeds=(0, 1, 2, 3, 4), car_factory=lambda: carla_like(noise=True), laps=1)`.
Later exercises compare against these five races; keep them.

The cell also gives the probe record's per-seed **five-track sums** of the same file (the lesson's objective, seeds 0 to 9).

Define:

- `noise_runs`: the `RepeatResult`; `one_lap_times`: its times as a list of floats; `one_lap_spread`: its spread (s);
- `shipped_by_seed`: a dict from race seed to that seed's race result (`noise_runs.results`, in seed order);
- `ticks`: each time as a whole number of ticks (`round(t / DT)`), and `on_ticks`: `True` when every time is within 1e-9 s of its ticks × `DT`;
- `sum_spread_1lap`, `sum_spread_3lap`: the spreads of `SUMS_ONE_LAP` and `SUMS_THREE_LAPS`, and `sum_to_track`: `sum_spread_1lap / one_lap_spread`;
- `fig`, `ax`: the five times against the race seed, with their median as a horizontal line, axis labels and a title;
- `why_sums`: two sentences: why is the five-track sum's spread bigger than one track's, and why does the lesson's search compare **per-seed sums** at two seeds by Unit 1's rule instead of one race against another?

**Prediction.** The lesson's shipped race at seed 0 is 96.40 s, and one track's 1-lap spread over five seeds should be a tick or two (0.05 to 0.15 s).
Every time is a whole number of ticks.
The record's five-track sums spread 0.45 s (1 lap) and 0.80 s (3 laps), so the sum's spread should be several times one track's: five tracks' noise adds up.

In [ ]:
# The probe record of plan roar-012 (p1): agents/unit-11.py, five training tracks, the sum per race seed (0-9), s.
SUMS_ONE_LAP = [476.80, 476.70, 477.00, 476.70, 476.60, 476.95, 476.55, 476.90, 476.65, 476.95]
SUMS_THREE_LAPS = [1383.25, 1383.05, 1383.25, 1382.85, 1382.95, 1383.45, 1382.75, 1383.15, 1383.15, 1383.55]

shipped_source = variant(UNIT_11, **NO_TIMING)
noise_runs = repeat_races(shipped_source, "test-circuit", n=5, seeds=(0, 1, 2, 3, 4),
                          car_factory=lambda: carla_like(noise=True), laps=1)
one_lap_times = [float(t) for t in noise_runs.times]
one_lap_spread = noise_runs.spread
shipped_by_seed = {seed: noise_runs.results[seed] for seed in range(5)}
ticks = [round(t / DT) for t in one_lap_times]
on_ticks = all(abs(one_lap_times[i] - ticks[i] * DT) <= 1e-9 for i in range(5))
sum_spread_1lap = max(SUMS_ONE_LAP) - min(SUMS_ONE_LAP)
sum_spread_3lap = max(SUMS_THREE_LAPS) - min(SUMS_THREE_LAPS)
sum_to_track = sum_spread_1lap / one_lap_spread

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(5), one_lap_times, "o-", label="unit-11.py, 1 lap")
ax.axhline(noise_runs.median, color="grey", linestyle="--", label=f"median {noise_runs.median:.2f} s")
ax.set_xlabel("race seed")
ax.set_ylabel("race time (s)")
ax.set_title("The objective's noise: unit-11.py on test-circuit, the same file at five race seeds")
ax.legend()
plt.show()
why_sums = (
    f"Each track adds its own noise of a tick or two, and the five tracks' noises do not cancel, so the five-track "
    f"1-lap sum spreads {sum_spread_1lap:.2f} s over ten seeds where one track spreads {one_lap_spread:.2f} s over "
    f"five ({sum_to_track:.1f} times as much). "
    "A setting's gain is measured against that noise: comparing one race with one race cannot tell a gain from a "
    "lucky seed, so the search races both settings at the same two seeds, sums each seed's five tracks, and accepts "
    "only when Unit 1's rule says the medians differ by more than the larger spread."
)

print("times:", one_lap_times, f"median {noise_runs.median:.2f} s, spread {one_lap_spread:.2f} s")
print("ticks:", ticks, "on ticks:", on_ticks)
print(f"sum spreads: 1 lap {sum_spread_1lap:.2f} s, 3 laps {sum_spread_3lap:.2f} s; 1-lap sum / one track {sum_to_track:.2f}")
print(why_sums)

# independent checks: the lesson's shipped race, and the record's spreads by hand
assert noise_runs.all_finished and noise_runs.collisions == 0 and all(clean(r) for r in noise_runs.results)
assert abs(one_lap_times[0] - 96.40) <= TICK  # the lesson's shipped race (U12.2)
assert ticks == [1928, 1928, 1928, 1927, 1926] and on_ticks
assert abs(one_lap_spread - 0.10) <= TICK and abs(one_lap_spread - (ticks[0] - ticks[-1]) * DT) <= 1e-9
assert abs(sum_spread_1lap - 0.45) <= 1e-9 and abs(sum_spread_3lap - 0.80) <= 1e-9
assert abs(sum_to_track - 4.5) <= 0.01 and sorted(shipped_by_seed) == [0, 1, 2, 3, 4]

## Exercise 2 - A knob map of three knobs

Classify three constants of `agents/unit-11.py` by racing them: `TARGET_SPEED = 50.0`, `LEAD_MIN_M = 8.0` and `K_E = 0.5`.
Race each on `test-circuit`, 1 lap, race seeds 0 and 1 (`race({knob: value}, seed=s)`): six races.
Compare each with Exercise 1's races of the shipped file at the same two seeds.

Define:

- `KNOBS`: given; `knob_races`: a dict from knob name to its list of two results (seeds 0, 1), and `knob_times`: the same with the race times;
- `shipped_two`: the shipped file's times at seeds 0 and 1 (from Exercise 1);
- `deltas`: a dict from knob name to the median of its two times minus the median of `shipped_two` (s);
- `beyond_noise`: a dict from knob name to `True` when Unit 1's rule says the two differ, either way (`noise_aware_better` in either order);
- `identical`: a dict from knob name to `True` when both its races are the same race (`same_race`) as the shipped race at the same seed;
- `verdict`: a dict from knob name to `"dead"` (identical), `"acts, beyond the noise"` or `"acts, within the noise"`;
- `k_e_reason`: one sentence: which part of the agent reads `K_E`, and why the shipped file never runs it;
- `sweep_limits`: two sentences: what can a two-seed, one-track, one-lap comparison miss?
  Use the lesson's findings on 1 lap against 3, and on five tracks against one.

**Prediction.** `K_E` is the PID steering's gain and the shipped file steers with MPC, so both its races should be the very same races as the shipped file's: dead.
`TARGET_SPEED = 50` lifts the profile's top speed, so it should act, by a second or so on one lap, beyond the noise.
`LEAD_MIN_M = 8` moved the five-track sum by only 0.77 s, so on one track it should act, but within the noise.

In [ ]:
KNOBS = {"TARGET_SPEED": 50.0, "LEAD_MIN_M": 8.0, "K_E": 0.5}

knob_races = {knob: [race({knob: value}, seed=seed) for seed in (0, 1)] for knob, value in KNOBS.items()}
knob_times = {knob: [result.total_time for result in results] for knob, results in knob_races.items()}
shipped_two = one_lap_times[:2]
deltas = {knob: float(np.median(times) - np.median(shipped_two)) for knob, times in knob_times.items()}
beyond_noise = {knob: noise_aware_better(times, shipped_two) or noise_aware_better(shipped_two, times)
                for knob, times in knob_times.items()}
identical = {knob: same_race(results[0], shipped_by_seed[0]) and same_race(results[1], shipped_by_seed[1])
             for knob, results in knob_races.items()}
verdict = {}
for knob in KNOBS:
    if identical[knob]:
        verdict[knob] = "dead"
    elif beyond_noise[knob]:
        verdict[knob] = "acts, beyond the noise"
    else:
        verdict[knob] = "acts, within the noise"
k_e_reason = (
    "K_E is the P gain of Unit 4's PID steering (the agent's steer_pid), which step() runs only when STEERING is "
    "'pid', and the shipped file steers with STEERING = 'mpc', so the PID is built but never called."
)
sweep_limits = (
    "Two seeds on one track and one lap show only what happens in the first 96 s on test-circuit: the lesson found "
    "1-lap and 3-lap rankings agree, but a setting can be clean for one lap and crash late (MPC_Q_E = 0.6 was clean on "
    "15 one-lap races and not clean at 3 laps), and TARGET_SPEED 50's gain grew from -1.4 s on five 1-lap tracks to "
    "-13.4 s over 3 laps. "
    "One track also misses what other tracks do: a knob can win on test-circuit and lose or crash elsewhere (p5's "
    "one-track setting was clean on test-circuit and on no other track), which is why the lesson's objective sums five "
    f"tracks, and why a {-deltas['LEAD_MIN_M']:.2f} s or {-deltas['TARGET_SPEED']:.2f} s gain at two seeds whose "
    "times tie is weak evidence."
)

for knob in KNOBS:
    print(f"{knob} = {KNOBS[knob]}: times {knob_times[knob]}, delta {deltas[knob]:+.2f} s, beyond the noise "
          f"{beyond_noise[knob]}, identical {identical[knob]}, clean {[clean(r) for r in knob_races[knob]]}: {verdict[knob]}")
print("shipped:", shipped_two)
print(k_e_reason)
print(sweep_limits)

assert all(abs(t - 96.40) <= TICK for t in shipped_two)
assert all(abs(a - b) <= TICK for a, b in zip(knob_times["TARGET_SPEED"], [96.15, 96.15]))
assert all(abs(a - b) <= TICK for a, b in zip(knob_times["LEAD_MIN_M"], [96.25, 96.20]))
assert all(clean(r) for results in knob_races.values() for r in results)
assert abs(deltas["TARGET_SPEED"] + 0.25) <= TICK and abs(deltas["LEAD_MIN_M"] + 0.175) <= TICK
assert deltas["K_E"] == 0.0
assert identical == {"TARGET_SPEED": False, "LEAD_MIN_M": False, "K_E": True}
assert verdict == {"TARGET_SPEED": "acts, beyond the noise", "LEAD_MIN_M": "acts, beyond the noise", "K_E": "dead"}

## Exercise 3 - A (1+2) evolution strategy, live

Run the lesson's evolution strategy at a size you can watch.

- **Knobs and box:** `MU_FRACTION` in [0.48, 0.66] and `A_DEC` in [4.0, 8.0], starting from `unit-11.py`'s 0.54 and 5.0.
- **Objective:** the sum of the 1-lap race times on the two tracks of `ES_TRACKS`, race seed 0.
  A setting is **rejected** if either of its two races is not clean.
- **Acceptance:** one race seed gives no spread, so Unit 1's rule cannot be used; instead a child replaces the parent when it is clean and its sum is lower than the parent's by more than `NOISE_BAND` (0.30 s, the hook's noise).
- **Children:** each generation draws two children from `rng = np.random.default_rng(3)`: for the first child, `MU_FRACTION` then `A_DEC`, then the same for the second child; each knob is the parent's value plus `rng.normal(0.0, sigma[knob])`, clipped to its box and rounded with `round(x, 2)`.
- **Step size:** `sigma` starts at one grid step (0.02 and 0.5); if the better clean child is accepted it becomes the parent and `sigma` is multiplied by 1.5, otherwise by 0.8.
- **Budget:** the parent's evaluation plus **two** generations: 10 races.

Write `evaluate(setting, tracks=ES_TRACKS, seed=0, source=UNIT_11)`: race the setting on each track with `max_seconds=200.0` (a clean lap takes about 96 s, so a crash loop is cut short) and return a dict with `"setting"`, `"times"` (a list, in track order), `"sum"`, `"clean"` (all races clean) and `"results"`.
Exercises 4 and 9 reuse it.

Define:

- `evaluate`: the function; `start_eval`: `evaluate` of the starting setting;
- `es_log`: a list with one dict per generation: `"children"` (their evaluations), `"accepted"` (`True` or `False`), `"sigma"` (after the update);
- `parent`, `parent_eval`: the final parent setting and its evaluation; `es_races`: the number of races raced (10);
- `es_gain`: `start_eval["sum"] - parent_eval["sum"]` (s);
- `fig`, `ax`: every evaluation's sum against the race count after it (not-clean evaluations drawn with a different marker), and the parent's sum as a step line, with a legend, axis labels and a title;
- `es_honest`: two sentences: after 10 races, would you call the final parent better than `unit-11.py`, and what did the record's (1+4)-ES do with 200 three-lap races (the lesson's p3)?

**Prediction.** Two generations of two children each is a tiny search: perhaps one accepted child, with `A_DEC` or `MU_FRACTION` a step up, a gain of a second or so on the 2-lap sum, or none at all.
A child with `MU_FRACTION` near 0.60 or above may not be clean.

In [ ]:
ES_TRACKS = ("test-circuit", "test-circuit-seed1")
ES_BOX = {"MU_FRACTION": (0.48, 0.66), "A_DEC": (4.0, 8.0)}
ES_START = {"MU_FRACTION": 0.54, "A_DEC": 5.0}
NOISE_BAND = 0.30  # s


def evaluate(setting, tracks=ES_TRACKS, seed=0, source=UNIT_11):
    results = [race(setting, track, seed=seed, source=source, max_seconds=200.0) for track in tracks]
    times = [result.total_time for result in results]
    return {"setting": dict(setting), "times": times, "sum": sum(times),
            "clean": all(clean(result) for result in results), "results": results}


start_eval = evaluate(ES_START)
es_races = 2
rng = np.random.default_rng(3)
sigma = {"MU_FRACTION": 0.02, "A_DEC": 0.5}  # one grid step each
parent, parent_eval = dict(ES_START), start_eval
history = [(es_races, start_eval)]  # (races after it, the evaluation)
parent_path = [(es_races, start_eval["sum"])]  # (races, the parent's sum)
es_log = []
for generation in range(2):
    children = []
    for _ in range(2):
        child = {}
        for knob in ("MU_FRACTION", "A_DEC"):
            low, high = ES_BOX[knob]
            child[knob] = round(min(max(parent[knob] + rng.normal(0.0, sigma[knob]), low), high), 2)
        children.append(child)
    evaluations = []
    for child in children:
        evaluations.append(evaluate(child))
        es_races += 2
        history.append((es_races, evaluations[-1]))
    clean_children = [e for e in evaluations if e["clean"]]
    accepted = False
    if clean_children:
        best = min(clean_children, key=lambda e: e["sum"])
        accepted = parent_eval["sum"] - best["sum"] > NOISE_BAND
    if accepted:
        parent, parent_eval = dict(best["setting"]), best
    factor = 1.5 if accepted else 0.8
    sigma = {knob: value * factor for knob, value in sigma.items()}
    es_log.append({"children": evaluations, "accepted": accepted, "sigma": dict(sigma)})
    parent_path.append((es_races, parent_eval["sum"]))
es_gain = start_eval["sum"] - parent_eval["sum"]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot([n for n, e in history if e["clean"]], [e["sum"] for n, e in history if e["clean"]], "o", label="clean evaluation")
ax.plot([n for n, e in history if not e["clean"]], [e["sum"] for n, e in history if not e["clean"]], "x", markersize=9,
        label="not clean (rejected)")
ax.step([n for n, _ in parent_path], [s for _, s in parent_path], where="post", label="the parent")
ax.set_xlabel("races so far")
ax.set_ylabel("sum of two 1-lap times (s)")
ax.set_title("A (1+2) evolution strategy, live: MU_FRACTION and A_DEC, race seed 0")
ax.legend()
plt.show()

print(f"start {start_eval['setting']}: {start_eval['times']}, sum {start_eval['sum']:.2f} s, clean {start_eval['clean']}")
for generation, entry in enumerate(es_log, start=1):
    for e in entry["children"]:
        print(f"  generation {generation}: {e['setting']} times {e['times']}, sum {e['sum']:.2f} s, clean {e['clean']}, "
              f"line errors {[round(line_error(r), 2) for r in e['results']]}")
    print(f"  accepted {entry['accepted']}, sigma {entry['sigma']}")
print(f"final parent {parent}: sum {parent_eval['sum']:.2f} s, gain {es_gain:.2f} s, after {es_races} races")
es_honest = (
    f"No: the final parent {parent} beat unit-11.py by {es_gain:.2f} s on one race seed, two tracks and one lap, "
    f"only {es_gain - NOISE_BAND:.2f} s more than the 0.30 s band, with no spread measured and no validation race, so it "
    "is one lucky draw as much as a gain, and in generation 2 a step away from it (0.57) was slower than unit-11.py. "
    "In the record's p3 the (1+4)-ES spent 200 three-lap races to cut the training objective from 1383.15 to 1357.17 s "
    "(25.98 s, less than coordinate search's 48.90 s), and its setting was clean on none of 5 races on the validation "
    "track (median 389.0 s against unit-11.py's 282.00 s)."
)
print(es_honest)

assert es_races == 10 and len(es_log) == 2 and all(len(entry["children"]) == 2 for entry in es_log)
assert start_eval["clean"] and all(abs(a - b) <= TICK for a, b in zip(start_eval["times"], [96.40, 95.20]))
assert same_race(start_eval["results"][0], shipped_by_seed[0])  # the same race as Exercise 1's, row for row
assert [e["setting"] for e in es_log[0]["children"]] == [{"MU_FRACTION": 0.58, "A_DEC": 4.0},
                                                        {"MU_FRACTION": 0.55, "A_DEC": 4.72}]
assert [e["setting"] for e in es_log[1]["children"]] == [{"MU_FRACTION": 0.57, "A_DEC": 4.0},
                                                        {"MU_FRACTION": 0.52, "A_DEC": 4.0}]
assert [entry["accepted"] for entry in es_log] == [True, False]
assert all(e["clean"] for entry in es_log for e in entry["children"])
assert abs(es_log[1]["sigma"]["MU_FRACTION"] - 0.02 * 1.5 * 0.8) <= 1e-12 and abs(es_log[1]["sigma"]["A_DEC"] - 0.6) <= 1e-12
assert parent == {"MU_FRACTION": 0.58, "A_DEC": 4.0} and abs(parent_eval["sum"] - 191.15) <= TICK
assert abs(es_gain - 0.45) <= TICK

## Exercise 4 - Random search, and why a not-clean race rejects

Draw four whole settings at random and evaluate each with Exercise 3's `evaluate` (two tracks, race seed 0): eight races.
Use `rng = np.random.default_rng(12)`; for each setting draw `MU_FRACTION` in [0.48, 0.66], then `A_DEC` in [4.0, 8.0], then `A_ACC` in [3.0, 6.0], each with `rng.uniform(low, high)` and rounded with `round(x, 2)`.
Compare every setting with Exercise 3's `start_eval` (`unit-11.py`).

Define:

- `draws`: the four settings (dicts), in the order drawn, and `random_evals`: their evaluations;
- `not_clean`: the settings whose evaluation is not clean;
- `faster_but_not_clean`: the settings that are not clean **and** whose sum is lower than `start_eval["sum"]`: the ones that would have been taken if a race that is not clean only counted its time;
- `best_random`: the clean setting with the lowest sum (`None` if no setting is clean), and `random_beats_start`: `True` when its sum is lower than `start_eval["sum"]` by more than `NOISE_BAND`;
- `fig`, `ax`: `MU_FRACTION` (x) against `A_ACC` (y) for the four draws, clean and not-clean settings with different markers, `unit-11.py`'s values marked too, with a legend, axis labels and a title;
- `why_random_is_slow`: two sentences: why did random search gain least of the three searches in the lesson's p3 record, and what does a not-clean but faster setting teach about the clean rule?

**Prediction.** Uniform draws over three knobs land anywhere in the box: some will have `A_ACC` near 3 (slow, about +3 s a track) and some a high `MU_FRACTION` or `A_DEC` (fast but perhaps not clean).
Most likely no clean draw beats `unit-11.py` by more than 0.30 s.

In [ ]:
RANDOM_BOX = {"MU_FRACTION": (0.48, 0.66), "A_DEC": (4.0, 8.0), "A_ACC": (3.0, 6.0)}

rng = np.random.default_rng(12)
draws, random_evals = [], []
for _ in range(4):
    draws.append({knob: round(rng.uniform(low, high), 2) for knob, (low, high) in RANDOM_BOX.items()})
for setting in draws:
    random_evals.append(evaluate(setting))
not_clean = [e["setting"] for e in random_evals if not e["clean"]]
faster_but_not_clean = [e["setting"] for e in random_evals if not e["clean"] and e["sum"] < start_eval["sum"]]
clean_evals = [e for e in random_evals if e["clean"]]
best_random, random_beats_start = None, False
if clean_evals:
    best_eval = min(clean_evals, key=lambda e: e["sum"])
    best_random = best_eval["setting"]
    random_beats_start = start_eval["sum"] - best_eval["sum"] > NOISE_BAND

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([e["setting"]["MU_FRACTION"] for e in random_evals if e["clean"]],
        [e["setting"]["A_ACC"] for e in random_evals if e["clean"]], "o", label="clean draw")
ax.plot([e["setting"]["MU_FRACTION"] for e in random_evals if not e["clean"]],
        [e["setting"]["A_ACC"] for e in random_evals if not e["clean"]], "x", markersize=9, label="not-clean draw")
ax.plot([0.54], [4.0], "*", markersize=14, label="unit-11.py")
ax.set_xlabel("MU_FRACTION")
ax.set_ylabel("A_ACC (m/s^2)")
ax.set_title("Random search: four draws in the box")
ax.legend()
plt.show()

for e in random_evals:
    print(f"{e['setting']}: times {e['times']}, sum {e['sum']:.2f} s ({e['sum'] - start_eval['sum']:+.2f} s), "
          f"clean {e['clean']}, collisions {[r.collisions for r in e['results']]}, "
          f"line errors {[round(line_error(r), 2) for r in e['results']]}")
print("not clean:", not_clean, "; faster but not clean:", faster_but_not_clean)
print("best random:", best_random, "; beats unit-11.py by more than the band:", random_beats_start)
why_random_is_slow = (
    "Random search throws away what it learns: every draw is a whole new setting anywhere in a six-knob box, so most "
    "of its draws land where some knob is far from its best (a low A_ACC or MU_FRACTION is slow, a high A_DEC is not "
    "clean), and in p3 its 200 races moved "
    "the objective only 1.62 s (1383.15 to 1381.53) where coordinate search, stepping from its incumbent, moved it 48.90 s; "
    f"here too one draw of four, A_DEC {not_clean[0]['A_DEC']}, crashed {sum(r.collisions for r in random_evals[0]['results'])} "
    "times and cost two 200 s races. "
    "No draw here was faster and not clean, but in p3 random search drew two (1354.45 s at 3.13 m and 1350.75 s at "
    "2.51 m off the line, with no collisions) that beat its own incumbent, so a rule that counted a not-clean race at its "
    "time would have accepted a car running wide of its line: the clean rule rejects them because the speed was bought "
    "with margin the race does not have."
)
print(why_random_is_slow)

assert draws == [{"MU_FRACTION": 0.53, "A_DEC": 7.79, "A_ACC": 3.57}, {"MU_FRACTION": 0.51, "A_DEC": 5.4, "A_ACC": 3.69},
                 {"MU_FRACTION": 0.6, "A_DEC": 4.46, "A_ACC": 5.69}, {"MU_FRACTION": 0.63, "A_DEC": 4.01, "A_ACC": 4.62}]
assert not_clean == [draws[0]] and faster_but_not_clean == []
assert [r.collisions for r in random_evals[0]["results"]] == [5, 7] and abs(random_evals[0]["sum"] - 400.0) <= TICK
assert [abs(e["sum"] - s) <= TICK for e, s in zip(random_evals[1:], [195.00, 188.30, 186.65])] == [True] * 3
assert best_random == draws[3] and random_beats_start

## Exercise 5 - A training win on the validation track

The lesson's live coordinate search ended its 40 races at `MU_FRACTION = 0.58` (`A_DEC` 5.0, `A_ACC` 4.0), 7.98 s faster than `unit-11.py` over five 1-lap training races (476.75 → 468.77 s).
Check it where it was never scored: race it on the validation track `test-circuit-seed2026`, **3 laps**, race seeds 0 and 1 (`race({"MU_FRACTION": 0.58}, VALIDATION, seed=s, laps=3)`): two races.
`unit-11.py`'s times there are given (the premise, U12.5a).

Define:

- `validation_races`: the two results; `validation_times`: their times; `validation_clean`: a list of `clean(result)`; `validation_max_error`: the largest `line_error` of the two (m);
- `beats_unit11`: `True` when both are clean and Unit 1's rule says they beat `UNIT_11_SEED2026` (the same seeds);
- `training_gain_per_lap`: (476.75 − 468.77) / 5, and `validation_gain_per_lap`: (the median of `UNIT_11_SEED2026` − the median of `validation_times`) / 3, both in seconds per lap;
- `still_a_test`: three sentences: you have now raced `MU_FRACTION = 0.58` on the validation track.
  If you ship it **because** of this result, can `test-circuit-seed2026` still tell you how it does on a track it has never seen?
  What would you race before shipping it (think of Unit 11's perturbed cars on this track, and the test track), and how many times would you race the test track?

**Prediction.** 0.58 was 1.60 s a lap faster per training track-lap set (7.98 s over five tracks), so about 1.6 s a lap... but the validation track has the lowest measured grip of all, where Unit 10's 0.58 was already near its edge (Unit 11 §7).
I expect it to be faster per lap than on training, or to run wide or crash: either way much less trustworthy than its training number.

In [ ]:
UNIT_11_SEED2026 = [282.05, 282.00]  # s: agents/unit-11.py on test-circuit-seed2026, 3 laps, race seeds 0 and 1

validation_races = [race({"MU_FRACTION": 0.58}, VALIDATION, seed=seed, laps=3) for seed in (0, 1)]
validation_times = [result.total_time for result in validation_races]
validation_clean = [clean(result) for result in validation_races]
validation_max_error = max(line_error(result) for result in validation_races)
beats_unit11 = all(validation_clean) and noise_aware_better(validation_times, UNIT_11_SEED2026)
training_gain_per_lap = (476.75 - 468.77) / 5
validation_gain_per_lap = (float(np.median(UNIT_11_SEED2026)) - float(np.median(validation_times))) / 3

for seed, result in zip((0, 1), validation_races):
    print(f"seed {seed}: {result.total_time:.2f} s, finished {result.finished}, collisions {result.collisions} "
          f"{[round(t, 2) for t in result.respawn_times]}, line error {line_error(result):.2f} m, clean {clean(result)}")
print(f"beats unit-11.py: {beats_unit11}; gain per lap: training {training_gain_per_lap:.3f} s, "
      f"validation {validation_gain_per_lap:.3f} s")
still_a_test = (
    f"No: 0.58 was {validation_gain_per_lap:.2f} s a lap faster there, clean at {validation_max_error:.2f} m, and if I "
    "ship it because of that, test-circuit-seed2026 has helped choose it and is no longer a track it has never seen, "
    "so a later race there can only confirm a choice it already made. "
    "Before shipping I would race it on Unit 11's perturbed cars on this track (the training cars 0-23 and then the "
    "held-out 24-47), because the lesson's crash-aware search showed 0.56 already crashing on low-grip car 8 on "
    "test-circuit, so 0.58's 1.6 s a lap may be bought with the margin Unit 11 kept. "
    "And I would race the test track test-circuit-seed2028 once, at the very end, after every choice is made, so "
    "that its answer is the honest one for a track nothing was tuned on."
)
print(still_a_test)

assert validation_clean == [True, True] and all(r.finished and r.collisions == 0 for r in validation_races)
assert all(abs(t - 276.85) <= TICK for t in validation_times)
assert 1.0 < validation_max_error <= CLEAN_ERROR and abs(validation_max_error - 1.26) <= 0.01
assert beats_unit11 and abs(training_gain_per_lap - 1.596) <= 1e-9
assert abs(validation_gain_per_lap - (282.025 - 276.85) / 3) <= TICK / 3

## Exercise 6 - The crash-aware check, and the margin rule

**Part A: the check.**
The crash-aware objective adds Unit 11's three lowest-grip training cars to every evaluation.
Race the speed search's setting (`SPEED_SETTING`, on `unit-11.py`) and `agents/unit-12.py` as shipped (`source=UNIT_12`, no constants changed) on `test-circuit` on each of cars 16, 18 and 8, race seed k mod 5, 1 lap, `max_seconds=250.0` (`race(setting, k=k, seed=k % 5, source=..., max_seconds=250.0)`): six races.

**Part B: the margin rule, no races.**
`CRASH_AWARE` is the lesson's table of the crash-aware search's ten evaluations (3 laps, 16 races each).
Unit 11's margin rule: find the **edge**, the highest value of a knob that is clean, and ship one grid step inside it.

Define:

- `low_grip_races`: a dict from `("speed", k)` and `("unit-12", k)` to the result, and `low_grip_clean`: the same with `clean(result)`;
- `speed_rejected`: `True` when any of the speed setting's three races is not clean (the crash-aware objective would reject it);
- `crash_costs`: a dict from each car on which the speed setting had a collision to its race time minus `unit-12.py`'s on the same car (s; a race cut at 250 s counts at 250 s);
- `clean_settings`: the settings in `CRASH_AWARE` with no race that was not clean; `best_clean`: the one with the lowest median of its two sums;
- `clean_neighbours`: the settings one grid step from `best_clean` on exactly one knob (steps 0.02, 0.5, 0.5) that are in `clean_settings`;
- `a_dec_edge`: the highest clean `A_DEC` with `MU_FRACTION` 0.54 and `A_ACC` 4.0, and `margin_setting`: the margin rule's setting along `A_DEC` (one grid step inside the edge, as a tuple); `margin_is_unit11`: `True` when it is `unit-11.py`'s (0.54, 5.0, 4.0);
- `which_to_race`: three sentences: the lesson measured that the edge (`unit-12.py`) is 1.35 to 1.40 s faster on the validation and test tracks and clean on 20 of Unit 11's 24 held-out cars, where `unit-11.py` is clean on 24.
  Which would you race, and why?
  What does your answer assume about how far the real car is from racekit's?

**Prediction.** The speed setting (0.60, 5.5, 4.5) asks for more grip than the low-grip cars have: it should crash on at least one of cars 16, 18, 8, at a cost of a large fraction of a lap or more each, while `unit-12.py` should be clean on all three (it is clean on the training family).
From the table: the clean settings are 0.54/5.0, 0.52/5.0, 0.54/5.5 and 0.54/4.5; the best is (0.54, 5.5, 4.0), whose only clean neighbour is `unit-11.py`; the `A_DEC` edge is 5.5 and the margin rule ships 5.0, `unit-11.py`.

In [ ]:
SPEED_SETTING = {"MU_FRACTION": 0.60, "A_DEC": 5.5, "A_ACC": 4.5}  # the speed search's result, on the grid
LOW_GRIP_CARS = (16, 18, 8)  # Unit 11's three lowest-grip training cars
# The lesson's table (probe p4): (MU_FRACTION, A_DEC, A_ACC) -> ((the two per-replicate sums, s), what was not clean)
CRASH_AWARE = {
    (0.54, 5.0, 4.0): ((2224.90, 2224.35), ""),
    (0.56, 5.0, 4.0): ((2411.70, 2205.10), "car 8"),
    (0.52, 5.0, 4.0): ((2245.50, 2245.25), ""),
    (0.54, 5.5, 4.0): ((2215.35, 2215.25), ""),
    (0.54, 4.5, 4.0): ((2238.75, 2238.90), ""),
    (0.54, 6.0, 4.0): ((2984.10, 2966.65), "cars 16, 18, 8, nominal test-circuit"),
    (0.54, 5.5, 4.5): ((2718.85, 2203.70), "cars 16, 8"),
    (0.54, 5.5, 3.5): ((2246.25, 2246.45), "cars 18, 8"),
    (0.56, 5.5, 4.0): ((2495.40, 2501.40), "cars 8, 16"),
    (0.52, 5.5, 4.0): ((2235.50, 2235.65), "car 18"),
}

# Part A: six races.
CAP = 250.0  # s
low_grip_races = {}
for k in LOW_GRIP_CARS:
    low_grip_races[("speed", k)] = race(SPEED_SETTING, k=k, seed=k % 5, max_seconds=CAP)
    low_grip_races[("unit-12", k)] = race({}, k=k, seed=k % 5, source=UNIT_12, max_seconds=CAP)
low_grip_clean = {key: clean(result) for key, result in low_grip_races.items()}
speed_rejected = not all(low_grip_clean[("speed", k)] for k in LOW_GRIP_CARS)


def counted(result):
    # A race cut at the cap counts at the cap.
    return result.total_time if result.finished else CAP


crash_costs = {k: counted(low_grip_races[("speed", k)]) - counted(low_grip_races[("unit-12", k)])
               for k in LOW_GRIP_CARS if low_grip_races[("speed", k)].collisions > 0}

# Part B: the margin rule on the table, no races.
STEPS = (0.02, 0.5, 0.5)
clean_settings = [s for s, (sums, bad) in CRASH_AWARE.items() if bad == ""]
best_clean = min(clean_settings, key=lambda s: np.median(CRASH_AWARE[s][0]))
neighbours = [tuple(round(best_clean[i] + d * STEPS[i], 2) if i == axis else best_clean[i] for i in range(3))
              for axis in range(3) for d in (+1, -1)]
clean_neighbours = [s for s in neighbours if s in clean_settings]
a_dec_edge = max(s[1] for s in clean_settings if s[0] == 0.54 and s[2] == 4.0)
margin_setting = (0.54, a_dec_edge - STEPS[1], 4.0)
margin_is_unit11 = margin_setting == (0.54, 5.0, 4.0)
which_to_race = (
    "I would race unit-11.py, the margin rule's answer: the edge buys about 1.4 s in a 270 to 280 s race (0.5 %), "
    "but it is not clean on 4 of 24 cars it never trained on, and on a car it does not fit, one crash costs far more than "
    f"1.4 s (here the speed setting's crashes cost {crash_costs[16]:.2f} s on car 16 and {crash_costs[8]:.2f} s on car 8 "
    "against unit-12.py on one lap). "
    "Racing the edge is a bet that 1.4 s is worth a one-in-six chance of running wide on a car like the held-out ones, "
    "and unit-11.py keeps one grid step of A_DEC in reserve. "
    "Either choice assumes the real car is within racekit's perturbed family; if the real car differs from the model by "
    "more than the family's spread (grip, throttle, lag), even unit-11.py's margin may not be enough, and the edge "
    "would be the first to fail."
)

for k in LOW_GRIP_CARS:
    for who in ("speed", "unit-12"):
        r = low_grip_races[(who, k)]
        print(f"car {k:2d} {who:8s}: {r.total_time:7.2f} s, finished {r.finished}, collisions {r.collisions} "
              f"{[round(t, 2) for t in r.respawn_times]}, line error {line_error(r):.2f} m, clean {clean(r)}")
print("speed setting rejected:", speed_rejected, "; crash costs:", {k: round(v, 2) for k, v in crash_costs.items()})
print("clean settings:", clean_settings)
print("best clean:", best_clean, "; clean neighbours:", clean_neighbours)
print("A_DEC edge:", a_dec_edge, "; margin rule:", margin_setting, "; is unit-11.py:", margin_is_unit11)
print(which_to_race)

assert low_grip_clean == {("speed", 16): False, ("unit-12", 16): True, ("speed", 18): True, ("unit-12", 18): True,
                         ("speed", 8): False, ("unit-12", 8): True}
assert speed_rejected and low_grip_races[("speed", 16)].collisions == 2 and low_grip_races[("speed", 8)].collisions == 2
assert not low_grip_races[("speed", 8)].finished and low_grip_races[("speed", 16)].finished
assert sorted(crash_costs) == [8, 16]
assert abs(crash_costs[16] - (213.85 - 96.05)) <= TICK and abs(crash_costs[8] - (250.0 - 96.25)) <= TICK
assert abs(low_grip_races[("speed", 18)].total_time - 94.95) <= TICK  # car 18: clean, and faster than unit-12.py
assert clean_settings == [(0.54, 5.0, 4.0), (0.52, 5.0, 4.0), (0.54, 5.5, 4.0), (0.54, 4.5, 4.0)]
assert best_clean == (0.54, 5.5, 4.0) and clean_neighbours == [(0.54, 5.0, 4.0)]
assert a_dec_edge == 5.5 and margin_setting == (0.54, 5.0, 4.0) and margin_is_unit11

## Exercise 7 - Profile a shorter horizon

Profile one race of `agents/unit-12.py` with `MPC_HORIZON = 20` (half the shipped 40): `test-circuit`, 1 lap, race seed 0, timing off.
Write the variant with `variant(UNIT_12, MPC_HORIZON=20, **NO_TIMING)` (you need its file name), then `profiler = cProfile.Profile()`, `profiler.enable()`, `run_race(...)` on `carla_like(noise=True)`, `profiler.disable()`.
Read it with `pstats.Stats(profiler)`: print the agent's functions sorted by cumulative time (`print_stats(path.stem, 12)`), then everything sorted by own time (`print_stats(8)`).

Your times depend on the machine and its load: **print** them and read them; do not compare them with a fixed number.

Define:

- `horizon_path`: the variant's path; `profiler`; `horizon_race`: the race result; `stats`: the `pstats.Stats`;
- `step_p50_ms`, `step_p99_ms`: `step()`'s 50th and 99th percentiles in this race (`horizon_race.step_ms`; the profiler slows them);
- `where_time_goes`: two sentences: which of the agent's functions takes the most cumulative time, about what share of `step()` is it here, and how does that compare with the lesson's p6 share at horizon 40 (`P6`)?
- `dead_work`: two sentences: what does `predict()` compute, who reads its answer, and what would removing it save (p6's `PREDICT_TICKS = 1` line)?
- `horizon_trade`: two sentences: from p6, what did halving the horizon buy and cost, and would you ship it for a race whose tick is 50 ms?

**Prediction.** `mpc_command()` (which calls `mpc()`) should still take the most of `step()`'s cumulative time, a smaller share than p6's 0.63 at horizon 40 because the solve is about half as long, while `predict()`'s fixed cost becomes a larger share.
Under the profiler every call is slower, so `step()`'s p50 will be well above p6's 0.537 ms.

In [ ]:
# The probe record of plan roar-012 (p6): agents/unit-11.py, test-circuit-seed1, 3 laps, without the profiler.
P6 = {
    "step() p50 / p99 (ms), MPC_HORIZON 40": (0.914, 0.995),
    "step() p50 (ms), MPC_HORIZON 20": 0.537,
    "step() p50 (ms), PREDICT_TICKS 1": 0.826,
    "step() p50 (ms), pure pursuit": 0.218,
    "mpc_command share of step(), MPC_HORIZON 40": 0.63,
    "predict share of step()": 0.19,
    "race time (s), MPC_HORIZON 40 / 20": (276.25, 276.35),
}

horizon_path = variant(UNIT_12, MPC_HORIZON=20, **NO_TIMING)
profiler = cProfile.Profile()
profiler.enable()
horizon_race = run_race(horizon_path, "test-circuit", car=carla_like(noise=True), seed=0, laps=1)
profiler.disable()
stats = pstats.Stats(profiler)
step_p50_ms = float(np.percentile(horizon_race.step_ms, 50))
step_p99_ms = float(np.percentile(horizon_race.step_ms, 99))
print(f"{horizon_race.total_time:.2f} s, clean {clean(horizon_race)}, {len(horizon_race.step_ms)} step() calls; "
      f"under the profiler step() p50 {step_p50_ms:.3f} ms, p99 {step_p99_ms:.3f} ms (this machine, this load)")
stats.sort_stats("cumulative").print_stats(horizon_path.stem, 12)  # the agent's own functions
stats.sort_stats("tottime").print_stats(8)  # everything, the agent and the simulator
where_time_goes = (
    "mpc_command() (with the mpc() solve under it) takes the most cumulative time of the agent's functions, about half "
    "of step()'s cumulative time in this profile (read off the printed cumtime column, mpc_command over step). "
    "At horizon 40 p6 measured 0.63, so halving the horizon shrinks the MPC's share, because the solve gets shorter "
    "while the rest of step() (predict(), measure()) costs the same."
)
dead_work = (
    "predict() rolls Unit 3's bicycle model PREDICT_TICKS = 20 ticks (1 s) ahead with the current command held and "
    "stores the answer in self.predicted_state, an instrument that nothing in step() reads, which is why PREDICT_TICKS is "
    "a dead knob (every race identical). "
    "Removing it would save its whole share of step() (p6: 19 % at horizon 40; it is the second-largest of the agent's "
    "functions in this profile, about a quarter of step() at horizon 20), and p6's PREDICT_TICKS = 1 "
    "line shows most of it: step()'s p50 fell from 0.914 to 0.826 ms with the same race."
)
horizon_trade = (
    "In p6 halving the horizon cut step()'s p50 from 0.914 to 0.537 ms (about 40 %), and it cost 0.10 s over 3 laps "
    "(276.25 to 276.35 s, two ticks), because the MPC plans only 1 s ahead instead of 2. "
    "With a 50 ms tick, step() at 0.9 ms uses under 2 % of the tick, so the saving buys nothing the race can use and I "
    "would not ship it: I would keep 40 and its 0.10 s."
)
print(where_time_goes)
print(dead_work)
print(horizon_trade)

# the race is deterministic (timing off), the profiler's numbers are not: only the race is asserted
assert clean(horizon_race) and abs(horizon_race.total_time - 95.95) <= TICK and len(horizon_race.step_ms) == 1919
assert horizon_path.stem in str(horizon_path) and step_p99_ms >= step_p50_ms > 0

## Exercise 8 - The traps, on purpose

**(a) Round trips.**
For each value in `VALUES`, check whether its `repr` reads back as exactly the same float, and whether its 3-decimal text (`f"{v:.3f}"`) does.

**(b) Accumulated steps.**
Start from `A_ACC = 4.0` and add the grid step 0.1 five times, one addition at a time: `accumulated`.
`snapped` is `round(accumulated, 2)`.
Race both as `A_ACC` on `test-circuit`, race seed 0, 1 lap: two races.

**(c) A dead knob.**
Race `K_HEAD = 2.0` on `test-circuit`, race seed 0, 1 lap (one race), and compare it with Exercise 1's shipped race at seed 0.

**(d) Ticks.**
`TIMES` are three race times as racekit returns them.

Define:

- `round_trips`, `three_decimals`: dicts from each value of `VALUES` to `True` or `False`;
- `accumulated`, `snapped`, `same_float` (`accumulated == snapped`);
- `accumulated_race`, `snapped_race`; `same_time`: their times equal; `same_telemetry`: `same_race` of the two;
- `dead_race`, `dead_identical`: `same_race` with the shipped race at seed 0;
- `time_ticks`: a dict from each time of `TIMES` to its whole number of ticks, and `printed`: a dict from each to its text with 2 decimals; `one_tick_apart`: `True` when the first two times are exactly one tick apart in ticks;
- `traps_summary`: three sentences: what goes wrong in (a), (b) and (c) if you are careless, and the one habit that prevents each.

**Prediction.** (a) Every `repr` reads back exactly; no 3-decimal text does (each of these values has more digits than three).
(b) Adding 0.1 five times gives a float just below 4.5, so `same_float` is `False`; the two races may have the same time but different telemetry, as in the lesson's trap 2.
(c) `K_HEAD` is read only by the PID steering, so its race is the shipped race row for row.
(d) 5593, 5592 and 1928 ticks; the first two print as 279.65 and 279.60, one tick apart.

In [ ]:
VALUES = (0.1 + 0.2, 0.54 + 3 * 0.02, 0.546666666606, 2 / 3)
TIMES = (279.65000000000003, 279.6, 96.4)  # s

# (a) round trips
round_trips = {v: float(repr(v)) == v for v in VALUES}
three_decimals = {v: float(f"{v:.3f}") == v for v in VALUES}
for v in VALUES:
    print(f"{v!r:22s} repr reads back: {round_trips[v]}; {v:.3f} reads back: {three_decimals[v]}")

# (b) accumulated steps
accumulated = 4.0
for _ in range(5):
    accumulated = accumulated + 0.1
snapped = round(accumulated, 2)
same_float = accumulated == snapped
accumulated_race = race({"A_ACC": accumulated})
snapped_race = race({"A_ACC": snapped})
same_time = accumulated_race.total_time == snapped_race.total_time
same_telemetry = same_race(accumulated_race, snapped_race)
print(f"accumulated {accumulated!r}, snapped {snapped!r}, the same float: {same_float}")
print(f"races: {accumulated_race.total_time:.2f} s and {snapped_race.total_time:.2f} s; the same time: {same_time}; "
      f"the same race: {same_telemetry}")

# (c) a dead knob
dead_race = race({"K_HEAD": 2.0})
dead_identical = same_race(dead_race, shipped_by_seed[0])
print(f"K_HEAD = 2.0: {dead_race.total_time:.2f} s; the shipped race row for row: {dead_identical}")

# (d) ticks
time_ticks = {t: round(t / DT) for t in TIMES}
printed = {t: f"{t:.2f}" for t in TIMES}
one_tick_apart = time_ticks[TIMES[0]] - time_ticks[TIMES[1]] == 1
for t in TIMES:
    print(f"{t!r:20s} {time_ticks[t]} ticks, printed {printed[t]}")
print("the first two one tick apart:", one_tick_apart)
traps_summary = (
    "(a) A constant written as rounded text is a different float (0.6000000000000001 written as 0.600 is 0.6), so a "
    "setting copied by hand races differently from the one measured; the habit is to write every constant with repr. "
    f"(b) Adding the step five times gives {accumulated!r}, not 4.5: the two races finish in the same "
    f"{snapped_race.total_time:.2f} s but are different races, so a search that does not snap can call one setting "
    "two and its record cannot be raced again from the grid value; the habit is to round every step to the grid "
    "(round(x, 2)). "
    "(c) K_HEAD is read only by the PID steering, so a sweep of it is the shipped race again and wastes races or, worse, "
    "reads noise as an effect; the habit is to check a knob is live (same_race against the shipped race) before searching it."
)
print(traps_summary)

assert all(round_trips.values()) and not any(three_decimals.values())
assert accumulated == 4.499999999999998 and snapped == 4.5 and not same_float
assert same_time and not same_telemetry and abs(snapped_race.total_time - 95.80) <= TICK
assert dead_identical and abs(dead_race.total_time - 96.40) <= TICK
assert time_ticks == {279.65000000000003: 5593, 279.6: 5592, 96.4: 1928}
assert printed == {279.65000000000003: "279.65", 279.6: "279.60", 96.4: "96.40"} and one_tick_apart

## Exercise 9 - Challenge: the speed loop and the top speed

The lesson's knob map found three big knobs the crash-aware search never touched: the speed loop's gains `SPEED_KP` and `SPEED_KI`, and the top speed `TARGET_SPEED`.
Search a small box of them by hand: each setting in `SPEED_BOX` (one knob changed), on `test-circuit`, race seeds 0 and 1, 1 lap: ten races.
Compare each with Exercise 1's shipped races at seeds 0 and 1.

Define:

- `speed_knob_races`: a dict from each label of `SPEED_BOX` to its two results;
- `speed_knob_table`: a dict from each label to `(median time minus the shipped median (s), both clean, the larger line_error of the two (m))`;
- `real_gains`: the labels that are clean in both races **and** beat the shipped file by Unit 1's rule;
- `margins`: a dict from each label to `CLEAN_ERROR` minus its larger line error (m): how far it stayed inside the clean rule;
- `fragile`: three sentences: are the gains you found safe to ship?
  Use your `margins`, and the lesson's records: in p2, `SPEED_KP = 0.3` was 9.35 s faster over five 1-lap tracks at a line error of 1.93 m, and in p3 the training-family search found `SPEED_KP = 0.4` and `TARGET_SPEED = 50` **not clean** at 3 laps.

**Prediction.** From p2, `SPEED_KP = 0.3` and `TARGET_SPEED = 50` and `55` should be faster by a second or more on one lap, `SPEED_KP = 0.4` a little faster, `SPEED_KI = 0.2` close to the noise.
The fast ones should run close to the clean rule's 2 m (p2's 1.93 m for `SPEED_KP = 0.3`).

In [ ]:
SPEED_BOX = {
    "SPEED_KP 0.3": {"SPEED_KP": 0.3},
    "SPEED_KP 0.4": {"SPEED_KP": 0.4},
    "SPEED_KI 0.2": {"SPEED_KI": 0.2},
    "TARGET_SPEED 50": {"TARGET_SPEED": 50.0},
    "TARGET_SPEED 55": {"TARGET_SPEED": 55.0},
}

speed_knob_races = {label: [race(setting, seed=seed) for seed in (0, 1)] for label, setting in SPEED_BOX.items()}
speed_knob_table, margins = {}, {}
real_gains = []
for label, results in speed_knob_races.items():
    times = [result.total_time for result in results]
    both_clean = all(clean(result) for result in results)
    worst = max(line_error(result) for result in results)
    speed_knob_table[label] = (float(np.median(times) - np.median(shipped_two)), both_clean, worst)
    margins[label] = CLEAN_ERROR - worst
    if both_clean and noise_aware_better(times, shipped_two):
        real_gains.append(label)

for label, (delta, both_clean, worst) in speed_knob_table.items():
    print(f"{label:16s} times {[r.total_time for r in speed_knob_races[label]]}, delta {delta:+.2f} s, "
          f"both clean {both_clean}, worst line error {worst:.2f} m, margin {margins[label]:.2f} m")
print("real gains:", real_gains)
fragile = (
    f"Not safely: all five settings beat the shipped file on these two seeds, but SPEED_KP 0.3, the biggest gain "
    f"({speed_knob_table['SPEED_KP 0.3'][0]:+.2f} s), kept only {margins['SPEED_KP 0.3']:.2f} m inside the 2 m rule, "
    "the least margin of the five, and the gains here are measured on one track and one lap where the shipped file's "
    "two races tie. "
    "In p2 the same SPEED_KP 0.3 ran 1.93 m off its line over five 1-lap tracks, only 0.07 m inside the rule, so one "
    "more track or lap is enough to cross it, and p3's training-family search found SPEED_KP 0.4 and TARGET_SPEED 50 "
    "not clean at 3 laps although both are clean here. "
    "Before shipping any of them I would race them at 3 laps on the five training tracks and on Unit 11's low-grip "
    "cars, and keep a grid step of margin as Unit 11's rule says."
)
print(fragile)

expected = {"SPEED_KP 0.3": (94.45, 94.45), "SPEED_KP 0.4": (95.60, 95.60), "SPEED_KI 0.2": (96.15, 96.10),
            "TARGET_SPEED 50": (96.15, 96.15), "TARGET_SPEED 55": (96.15, 96.15)}
for label, pair in expected.items():
    assert all(abs(r.total_time - t) <= TICK for r, t in zip(speed_knob_races[label], pair)), label
    assert speed_knob_table[label][1]
assert abs(speed_knob_table["SPEED_KP 0.3"][0] + 1.95) <= TICK and abs(margins["SPEED_KP 0.3"] - 0.38) <= 0.01
assert min(margins, key=margins.get) == "SPEED_KP 0.3"
assert real_gains == list(SPEED_BOX)
assert same_race(speed_knob_races["TARGET_SPEED 50"][0], knob_races["TARGET_SPEED"][0])  # Exercise 2's race again

## Exercise 10 - Challenge: policy gradient against coordinate search, same budget

The lesson's optional part taught the one-knob policy gradient on a pretend race.
Race it for real, against coordinate search, on **the same race budget**: at most 5 races each.

- **The knob and the race:** `A_ACC`, on `test-circuit`, 1 lap, race seed 0 for every race (one seed: the noise is the same fixed draw for every setting, so this compares the two methods, not settings across seeds).
- **Policy gradient (5 races):** μ = 4.0, σ = 0.25, α = 0.2, `pg_rng = np.random.default_rng(10)`.
  Two updates; each draws N = 2 values `x = round(mu + pg_rng.normal(0.0, sigma), 2)`, races each as `A_ACC`, and updates μ ← μ + α · mean over the draws of (b − Tᵢ) · (xᵢ − μ) / σ², with b the mean of the two times.
  Then race the final μ (rounded to 2 decimals): the policy's answer.
  That is 2 × 2 + 1 = 5 races.
- **Coordinate search (at most 5 races; it stops early when a step no longer wins):** start at 4.0 (Exercise 1's shipped race at seed 0 **is** this race: reuse it, and count it), then 4.5, then 3.5; take the faster of the two if it beats the incumbent by more than `BAND` (0.10 s); then keep stepping (0.5) in that direction while each step beats the incumbent by more than `BAND`, until 5 races have been used.

Define:

- `pg_log`: a list with one dict per update: `"x"` (the two values raced), `"times"`, `"baseline"`, `"mu"` (after the update); `pg_answer`: the final μ rounded, and `pg_answer_time`: its race time;
- `coord_log`: a list of `(A_ACC, race time)` in the order raced (at most 5 entries, the first from Exercise 1); `coord_answer`, `coord_answer_time`: the incumbent at the end and its time;
- `winner`: `"policy gradient"`, `"coordinate search"` or `"tie"` (the answers' times within one tick);
- `honest_comparison`: three sentences: which method gave the better answer on this budget, what would you need to race before believing it is better on other seeds and tracks, and why is a policy gradient built for a different kind of problem than tuning one constant?

**Prediction.** Coordinate search should step up from 4.0 to 4.5 and perhaps 5.0 (p2's best `A_ACC` is 4.5).
With only two updates of two draws each and σ = 0.25, the policy's μ should move only a little from 4.0, maybe toward 4.5, so coordinate search should give the better answer on this budget, or tie.

In [ ]:
BAND = 0.10  # s: one track, one seed, 1 lap

# Policy gradient: 2 updates of 2 draws, then the final mu: 5 races.
pg_rng = np.random.default_rng(10)
mu, sigma_pg, alpha = 4.0, 0.25, 0.2
pg_log = []
for update in range(2):
    x = [round(mu + pg_rng.normal(0.0, sigma_pg), 2) for _ in range(2)]
    times = [race({"A_ACC": value}).total_time for value in x]
    baseline = float(np.mean(times))
    gradient = float(np.mean([(baseline - times[i]) * (x[i] - mu) / sigma_pg ** 2 for i in range(2)]))
    mu = mu + alpha * gradient
    pg_log.append({"x": x, "times": times, "baseline": baseline, "mu": mu})
pg_answer = round(mu, 2)
pg_answer_time = race({"A_ACC": pg_answer}).total_time

# Coordinate search: 4.0 (Exercise 1's race at seed 0), 4.5, 3.5, then onward in the winning direction: at most 5 races.
coord_log = [(4.0, shipped_by_seed[0].total_time)]
for value in (4.5, 3.5):
    coord_log.append((value, race({"A_ACC": value}).total_time))
coord_answer, coord_answer_time = coord_log[0]
direction = 0
faster = min(coord_log[1:], key=lambda entry: entry[1])
if coord_answer_time - faster[1] > BAND:
    coord_answer, coord_answer_time = faster
    direction = 1 if faster[0] > 4.0 else -1
while direction != 0 and len(coord_log) < 5:
    candidate = round(coord_answer + direction * 0.5, 2)
    candidate_time = race({"A_ACC": candidate}).total_time
    coord_log.append((candidate, candidate_time))
    if coord_answer_time - candidate_time > BAND:
        coord_answer, coord_answer_time = candidate, candidate_time
    else:
        direction = 0

if abs(pg_answer_time - coord_answer_time) <= TICK:
    winner = "tie"
elif pg_answer_time < coord_answer_time:
    winner = "policy gradient"
else:
    winner = "coordinate search"

for update, entry in enumerate(pg_log, start=1):
    print(f"update {update}: x {entry['x']}, times {entry['times']}, baseline {entry['baseline']:.3f} s, "
          f"mu {entry['mu']:.4f}")
print(f"policy gradient: A_ACC {pg_answer} in {pg_answer_time:.2f} s (5 races)")
print("coordinate search:", coord_log, f"-> A_ACC {coord_answer} in {coord_answer_time:.2f} s ({len(coord_log)} races)")
print("winner:", winner)
honest_comparison = (
    f"Coordinate search gave the better answer here: A_ACC {coord_answer} in {coord_answer_time:.2f} s against the "
    f"policy's {pg_answer} in {pg_answer_time:.2f} s, and it used only {len(coord_log)} of its 5 races, because its "
    "third step (5.0) did not win and the rule stops there, while the policy's two small updates moved mu only from 4.0 "
    f"to {mu:.2f}. "
    "These are single races at one seed on one track, where the noise is one fixed draw, so before believing either "
    "method is better I would race both answers at several seeds on the five training tracks and at 3 laps, and repeat "
    "each method with other random draws (pg_rng) to see how much the policy's answer itself varies. "
    "A policy gradient is built for problems where the thing being learned is a whole policy with many parameters "
    "and a reward that only comes back noisily from many tries, where no grid can be stepped; for one constant on a "
    "grid of a few values, a direct search that races each value is simpler and spends the races better."
)
print(honest_comparison)

assert [entry["x"] for entry in pg_log] == [[3.72, 3.82], [3.82, 4.09]]
assert all(abs(a - b) <= TICK for a, b in zip(pg_log[0]["times"] + pg_log[1]["times"], [97.05, 96.80, 96.80, 96.20]))
assert abs(pg_log[0]["mu"] - 4.02) <= 1e-6 and pg_answer == 4.15 and abs(pg_answer_time - 96.10) <= TICK
assert [value for value, _ in coord_log] == [4.0, 4.5, 3.5, 5.0]
assert all(abs(t - s) <= TICK for (_, t), s in zip(coord_log, [96.40, 95.80, 97.65, 96.35]))
assert coord_answer == 4.5 and abs(coord_answer_time - 95.80) <= TICK and winner == "coordinate search"